# A simple 3-D exploration of auxiliary balance in GFS

This notebook asks whether probability segments can be rearranged while preserving:

1. each unit's first-order inclusion probability (FIP) $\pi_k$,
2. a fixed sample size $n$, and
3. exact or approximate Horvitz--Thompson (HT) balance on an auxiliary variable $x$.

The code deliberately uses small arrays, explicit loops, and short helper functions. A configuration is just a table of segments: `(unit, lower endpoint, upper endpoint)`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

## 1. A very small population

The height attached to unit $k$ is $h_k=x_k/\pi_k$. For a sample $S$, adding the intersected heights gives the HT estimate $\widehat X_{HT}(S)$.

In [ ]:
N = 8
n = 4
pi = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
h = x / pi
X = x.sum()

population = pd.DataFrame({"unit": np.arange(1, N + 1), "pi": pi, "x": x, "h=x/pi": h})
display(population)
print("N =", N)
print("n =", n)
print("X = sum(x) =", X)
print("sum(pi) =", pi.sum(), "= n:", np.isclose(pi.sum(), n))

## 2. Manually construct an exactly balanced design

$S_1=\{1,2,7,8\}$ occupies the lower half of the randomization axis and $S_2=\{3,4,5,6\}$ occupies the upper half. Thus every unit has exactly one segment of length $0.5$.

In [ ]:
S1 = [1, 2, 7, 8]
S2 = [3, 4, 5, 6]

balanced_segments = []
for k in S1:
    balanced_segments.append([k, 0.0, 0.5])
for k in S2:
    balanced_segments.append([k, 0.5, 1.0])
balanced_segments = pd.DataFrame(balanced_segments, columns=["unit", "lower", "upper"])
balanced_segments["length"] = balanced_segments["upper"] - balanced_segments["lower"]
display(balanced_segments.sort_values("unit"))

In [ ]:
def calculate_HT_x(sample, x, pi):
    total = 0.0
    for k in sample:
        total += x[k - 1] / pi[k - 1]
    return total

for name, sample in [("S1", S1), ("S2", S2)]:
    terms = [x[k - 1] / pi[k - 1] for k in sample]
    print(name, "=", sample)
    print("  heights =", terms)
    print("  HT_x =", sum(terms), "; X =", X, "; error =", sum(terms) - X)

Both samples contain four units and both HT totals equal $X=36$. FIP, fixed size, and balance are three separate checks.

## 3. The ordinary 2-D GFS picture

A horizontal line selects every segment containing $r$. Endpoints use the convention `lower <= r < upper`, except that `r=1` is treated as the top limit.

In [ ]:
def get_sample_at_r(segments, r):
    sample = []
    r_check = min(r, np.nextafter(1.0, 0.0))
    for _, row in segments.iterrows():
        if row["lower"] <= r_check < row["upper"]:
            sample.append(int(row["unit"]))
    return sorted(sample)

def calculate_balance_error(sample, x, pi):
    return calculate_HT_x(sample, x, pi) - x.sum()

def plot_gfs_2d(segments, r, title="2-D GFS configuration"):
    fig, ax = plt.subplots(figsize=(10, 5))
    sample = get_sample_at_r(segments, r)
    for _, row in segments.iterrows():
        k = int(row["unit"])
        color = "tab:orange" if k in sample else "tab:blue"
        ax.plot([k, k], [row["lower"], row["upper"]], lw=12, color=color, solid_capstyle="butt")
    ax.axhline(r, color="black", ls="--", lw=2, label=f"r = {r}")
    ax.set(xlabel="population unit k", ylabel="randomization coordinate r", xlim=(0.4, N + 0.6), ylim=(0, 1))
    ax.set_xticks(range(1, N + 1))
    ax.grid(axis="y", alpha=0.25)
    ax.set_title(title + f"\nselected sample = {sample}")
    ax.legend()
    plt.show()

plot_gfs_2d(balanced_segments, 0.25, "Balanced design")
plot_gfs_2d(balanced_segments, 0.75, "Balanced design")

At $r=0.25$ the line intersects $S_1$; at $r=0.75$ it intersects $S_2$.

## 4. A 3-D representation

Each probability segment becomes a prism. Its length in the $r$ direction is its probability length, and its height is $h_k=x_k/\pi_k$. Orange prisms are intersected by the transparent random plane.

In [ ]:
def plot_gfs_3d(segments, r, x, pi, title="3-D GFS configuration"):
    fig = plt.figure(figsize=(11, 7))
    ax = fig.add_subplot(111, projection="3d")
    sample = get_sample_at_r(segments, r)
    heights = x / pi

    for _, row in segments.iterrows():
        k = int(row["unit"])
        color = "tab:orange" if k in sample else "tab:blue"
        ax.bar3d(k - 0.35, row["lower"], 0, 0.7, row["upper"] - row["lower"],
                 heights[k - 1], color=color, alpha=0.8, shade=True)

    xx, zz = np.meshgrid([0.5, N + 0.5], [0, heights.max() * 1.05])
    yy = np.full_like(xx, r)
    ax.plot_surface(xx, yy, zz, color="gray", alpha=0.22)
    ax.set(xlabel="unit k", ylabel="randomization r", zlabel="height h = x/pi")
    ax.set_xticks(range(1, N + 1))
    ax.set_title(title + f"\nplane r={r}, selected={sample}")
    ax.view_init(elev=24, azim=-55)
    plt.show()

r = 0.25
selected = get_sample_at_r(balanced_segments, r)
print("S(r) =", selected)
print("sample size =", len(selected))
print("sum x_k/pi_k =", calculate_HT_x(selected, x, pi))
print("X =", X)
print("balance error =", calculate_balance_error(selected, x, pi))
plot_gfs_3d(balanced_segments, r, x, pi, "Balanced design")

The 3-D picture is an intuition: probability length controls FIP, the number of intersected prisms controls sample size, and the sum of their heights is the HT total.

## 5. Turn the configuration into strips

All unique segment endpoints divide $[0,1]$ into strips. The sample is constant inside each open strip, so its midpoint is enough to identify the sample.

In [ ]:
def design_strips(segments, x, pi):
    endpoints = sorted(set(segments["lower"]).union(set(segments["upper"])))
    rows = []
    X = x.sum()
    for j in range(len(endpoints) - 1):
        lower, upper = endpoints[j], endpoints[j + 1]
        midpoint = (lower + upper) / 2
        sample = get_sample_at_r(segments, midpoint)
        HT_x = calculate_HT_x(sample, x, pi)
        rows.append([j + 1, lower, upper, upper - lower, sample, len(sample), HT_x, X, HT_x - X])
    columns = ["strip", "lower", "upper", "probability", "sample", "sample_size", "HT_x", "X", "balance_error"]
    return pd.DataFrame(rows, columns=columns)

balanced_strips = design_strips(balanced_segments, x, pi)
display(balanced_strips)

Each row is a possible sample and its design probability. Both errors are zero here.

## 6. Two simple balance criteria

$$C_2=\sum_s p_s(\widehat X_{HT}(s)-X)^2,\qquad
C_{\max}=\max_s|\widehat X_{HT}(s)-X|.$$

$C_2$ is average squared imbalance; $C_{\max}$ is the worst imbalance among possible samples.

In [ ]:
def evaluate_design(segments, x, pi):
    strips = design_strips(segments, x, pi)
    C2 = np.sum(strips["probability"] * strips["balance_error"] ** 2)
    Cmax = np.max(np.abs(strips["balance_error"]))
    return C2, Cmax, strips

balanced_C2, balanced_Cmax, _ = evaluate_design(balanced_segments, x, pi)
print("Balanced design: C2 =", balanced_C2, "; Cmax =", balanced_Cmax)

## 7. A fixed-size but unbalanced configuration

Now the lower strip contains $\{1,2,3,4\}$ and the upper strip its complement. Every unit still occupies length $0.5$, and both samples still have size four, but their $x$ totals differ.

In [ ]:
U1 = [1, 2, 3, 4]
U2 = [5, 6, 7, 8]
unbalanced_segments = []
for k in U1:
    unbalanced_segments.append([k, 0.0, 0.5])
for k in U2:
    unbalanced_segments.append([k, 0.5, 1.0])
unbalanced_segments = pd.DataFrame(unbalanced_segments, columns=["unit", "lower", "upper"])
unbalanced_segments["length"] = unbalanced_segments["upper"] - unbalanced_segments["lower"]

fip_check = unbalanced_segments.groupby("unit")["length"].sum().reindex(range(1, N + 1))
print("FIP reconstructed from segment lengths:", fip_check.to_list())
unbalanced_C2, unbalanced_Cmax, unbalanced_strips = evaluate_design(unbalanced_segments, x, pi)
display(unbalanced_strips)
print("Unbalanced design: C2 =", unbalanced_C2, "; Cmax =", unbalanced_Cmax)
print("Balanced design:   C2 =", balanced_C2, "; Cmax =", balanced_Cmax)

In [ ]:
plot_gfs_2d(unbalanced_segments, 0.25, "Unbalanced design")
plot_gfs_3d(unbalanced_segments, 0.25, x, pi, "Unbalanced design")

The comparison isolates the main point: the same FIP and the same fixed sample size do **not** guarantee auxiliary balance.

## 8. One-for-one exchange intuition

If $k\in s_i\setminus s_j$ and $l\in s_j\setminus s_i$, exchanging them changes the HT total in strip $i$ by $h_l-h_k$, and in strip $j$ by $h_k-h_l$. Sample sizes remain unchanged. With equal strip widths, each unit merely moves from one strip to the other, so its total segment length (FIP) also remains unchanged.

In [ ]:
def list_one_for_one_exchanges(sample_i, sample_j, x, pi):
    only_i = sorted(set(sample_i) - set(sample_j))
    only_j = sorted(set(sample_j) - set(sample_i))
    rows = []
    for k in only_i:
        for l in only_j:
            h_k = x[k - 1] / pi[k - 1]
            h_l = x[l - 1] / pi[l - 1]
            rows.append([k, l, h_k, h_l, abs(h_k - h_l)])
    return pd.DataFrame(rows, columns=["k", "l", "h_k", "h_l", "absolute_difference"]).sort_values("absolute_difference")

one_swaps = list_one_for_one_exchanges(U1, U2, x, pi)
display(one_swaps)

k, l = 4, 5
before_i = calculate_HT_x(U1, x, pi)
after_i = before_i - h[k - 1] + h[l - 1]
print("Example swap k=4 with l=5")
print("Change predicted by h_l - h_k =", h[l - 1] - h[k - 1])
print("HT in lower strip:", before_i, "->", after_i)

Similar heights make a small local change, so such exchanges tend to disturb an already balanced design less. Improving a badly unbalanced design can instead require a larger, well-directed change.

## 9. Two-for-two group exchanges

Equal group sizes preserve fixed sample size. Exact balance is preserved when the two exchanged groups have equal height sums.

In [ ]:
def list_two_for_two_exchanges(sample_i, sample_j, x, pi):
    only_i = sorted(set(sample_i) - set(sample_j))
    only_j = sorted(set(sample_j) - set(sample_i))
    rows = []
    for A in combinations(only_i, 2):
        for B in combinations(only_j, 2):
            sum_A = sum(x[k - 1] / pi[k - 1] for k in A)
            sum_B = sum(x[k - 1] / pi[k - 1] for k in B)
            rows.append([A, B, sum_A, sum_B, abs(sum_A - sum_B)])
    columns = ["A", "B", "sum_h_A", "sum_h_B", "balance_difference"]
    return pd.DataFrame(rows, columns=columns).sort_values("balance_difference")

two_swaps = list_two_for_two_exchanges(U1, U2, x, pi)
display(two_swaps.head(10))

## 10. One local-improvement step

We try every one-for-one swap between the two strips. For each candidate we rebuild the two-strip configuration and recompute $C_2$ and $C_{\max}$. This is an exhaustive check of a tiny neighborhood, not an advanced optimizer.

In [ ]:
def two_strip_configuration(lower_sample, upper_sample):
    rows = []
    for k in lower_sample:
        rows.append([k, 0.0, 0.5])
    for k in upper_sample:
        rows.append([k, 0.5, 1.0])
    result = pd.DataFrame(rows, columns=["unit", "lower", "upper"])
    result["length"] = result["upper"] - result["lower"]
    return result

candidate_rows = []
for k in U1:
    for l in U2:
        new_lower = sorted((set(U1) - {k}) | {l})
        new_upper = sorted((set(U2) - {l}) | {k})
        candidate = two_strip_configuration(new_lower, new_upper)
        new_C2, new_Cmax, _ = evaluate_design(candidate, x, pi)
        candidate_rows.append([k, l, new_lower, new_upper, new_C2, new_Cmax])

candidates = pd.DataFrame(candidate_rows, columns=["k", "l", "lower_sample", "upper_sample", "C2", "Cmax"])
candidates = candidates.sort_values(["C2", "Cmax", "k", "l"]).reset_index(drop=True)
display(candidates)
best = candidates.iloc[0]
after_segments = two_strip_configuration(best["lower_sample"], best["upper_sample"])
after_C2, after_Cmax, after_strips = evaluate_design(after_segments, x, pi)

In [ ]:
print("BEFORE")
print("samples:", U1, "and", U2)
print("C2 =", unbalanced_C2, "; Cmax =", unbalanced_Cmax)
display(unbalanced_strips)
plot_gfs_2d(unbalanced_segments, 0.25, "BEFORE: unbalanced")
plot_gfs_3d(unbalanced_segments, 0.25, x, pi, "BEFORE: unbalanced")

print("AFTER one best swap: k =", int(best["k"]), "with l =", int(best["l"]))
print("samples:", best["lower_sample"], "and", best["upper_sample"])
print("C2 =", after_C2, "; Cmax =", after_Cmax)
display(after_strips)
plot_gfs_2d(after_segments, 0.25, "AFTER: one best swap")
plot_gfs_3d(after_segments, 0.25, x, pi, "AFTER: one best swap")

For this example one exchange reduces the squared imbalance dramatically. It does not necessarily reach exact balance in one move.

### Optional: repeat the same simple step

The following loop accepts only a strict improvement and stops when none exists. Set `maximum_iterations` to a small number.

In [ ]:
current_lower = U1.copy()
current_upper = U2.copy()
maximum_iterations = 10
history = []

for iteration in range(maximum_iterations):
    current_segments = two_strip_configuration(current_lower, current_upper)
    current_C2, current_Cmax, _ = evaluate_design(current_segments, x, pi)
    best_move = None
    for k in current_lower:
        for l in current_upper:
            trial_lower = sorted((set(current_lower) - {k}) | {l})
            trial_upper = sorted((set(current_upper) - {l}) | {k})
            trial_segments = two_strip_configuration(trial_lower, trial_upper)
            trial_C2, trial_Cmax, _ = evaluate_design(trial_segments, x, pi)
            if best_move is None or trial_C2 < best_move[0]:
                best_move = [trial_C2, trial_Cmax, k, l, trial_lower, trial_upper]
    history.append([iteration, current_lower, current_upper, current_C2, current_Cmax])
    if best_move[0] >= current_C2:
        break
    current_lower, current_upper = best_move[4], best_move[5]

final_segments = two_strip_configuration(current_lower, current_upper)
final_C2, final_Cmax, _ = evaluate_design(final_segments, x, pi)
history.append([len(history), current_lower, current_upper, final_C2, final_Cmax])
display(pd.DataFrame(history, columns=["iteration", "lower_sample", "upper_sample", "C2", "Cmax"]))

## 11. Important special case: PPS

If $\pi_k=cx_k$, then every height is $x_k/\pi_k=1/c$. Consequently any fixed-size sample has HT total $n/c$. Because $c=n/X$, this equals $X$.

We choose a small positive `x_pps` for which all resulting probabilities are at most one.

In [ ]:
x_pps = np.array([1, 2, 2, 3, 3, 4, 4, 5], dtype=float)
n_pps = 4
X_pps = x_pps.sum()
c = n_pps / X_pps
pi_pps = c * x_pps
h_pps = x_pps / pi_pps

print("c = n/X =", c)
print("pi =", pi_pps)
print("sum(pi) =", pi_pps.sum(), "; all pi <= 1:", np.all(pi_pps <= 1))
print("h = x/pi =", h_pps)

example_sample = [1, 3, 6, 8]
print("example fixed-size sample =", example_sample)
print("HT_x =", calculate_HT_x(example_sample, x_pps, pi_pps))
print("n/c =", n_pps / c, "; X =", X_pps)

Thus PPS plus fixed sample size is automatically exactly balanced on that same $x$. The more interesting problem is when FIP come from one requirement and balance is imposed using another auxiliary variable.

## 12. Two auxiliary variables

For $x_{1k}$ and $x_{2k}$, attach the vector $(x_{1k}/\pi_k, x_{2k}/\pi_k)$ to unit $k$. No 4-D plot is needed: evaluate both HT errors together.

In [ ]:
x1 = x.astype(float)
x2 = np.array([2, 1, 4, 3, 6, 5, 8, 7], dtype=float)
X1, X2 = x1.sum(), x2.sum()

def evaluate_two_variables(segments, x1, x2, pi):
    endpoints = sorted(set(segments["lower"]).union(set(segments["upper"])))
    rows = []
    C = 0.0
    for j in range(len(endpoints) - 1):
        lower, upper = endpoints[j], endpoints[j + 1]
        probability = upper - lower
        sample = get_sample_at_r(segments, (lower + upper) / 2)
        HT1 = calculate_HT_x(sample, x1, pi)
        HT2 = calculate_HT_x(sample, x2, pi)
        error1, error2 = HT1 - x1.sum(), HT2 - x2.sum()
        C += probability * (error1 ** 2 + error2 ** 2)
        rows.append([sample, probability, HT1, error1, HT2, error2])
    columns = ["sample", "probability", "HT_X1", "error_X1", "HT_X2", "error_X2"]
    return C, pd.DataFrame(rows, columns=columns)

C_two, two_variable_table = evaluate_two_variables(balanced_segments, x1, x2, pi)
display(two_variable_table)
print("Combined criterion C =", C_two)

The 3-D construction is mainly visual intuition. Algebraically, the same GFS configuration can be judged against several auxiliary totals at once.

## 13. Final editable experiment

Edit the values below. Then construct `my_segments` by listing each unit's segment endpoints. A unit may appear in more than one row if its total probability is split across several segments. The total length for unit $k$ should equal `pi[k-1]`.

In [ ]:
# ===== EDIT HERE =====

N = 8
n = 4
pi = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
r = 0.25

# Each row is: [unit, lower endpoint, upper endpoint]
my_segments = pd.DataFrame([
    [1, 0.0, 0.5],
    [2, 0.0, 0.5],
    [7, 0.0, 0.5],
    [8, 0.0, 0.5],
    [3, 0.5, 1.0],
    [4, 0.5, 1.0],
    [5, 0.5, 1.0],
    [6, 0.5, 1.0],
], columns=["unit", "lower", "upper"])
my_segments["length"] = my_segments["upper"] - my_segments["lower"]

In [ ]:
# Three separate checks
my_fip = my_segments.groupby("unit")["length"].sum().reindex(range(1, N + 1), fill_value=0).to_numpy()
my_C2, my_Cmax, my_strips = evaluate_design(my_segments, x, pi)

print("Prescribed pi:              ", pi)
print("Lengths reconstructed as FIP:", my_fip)
print("FIP preserved:", np.allclose(my_fip, pi))
print("Every strip has size n:", np.all(my_strips["sample_size"] == n))
print("C2 =", my_C2, "; Cmax =", my_Cmax)
display(my_strips)

my_sample = get_sample_at_r(my_segments, r)
print("At r =", r, ": sample =", my_sample)
print("HT_x =", calculate_HT_x(my_sample, x, pi))
print("balance error =", calculate_balance_error(my_sample, x, pi))
plot_gfs_2d(my_segments, r, "Your editable configuration")
plot_gfs_3d(my_segments, r, x, pi, "Your editable configuration")

## Takeaway

Segment rearrangement gives a search space of sampling designs. Segment lengths enforce prescribed FIP; strip intersection counts enforce fixed sample size; HT errors measure auxiliary balance. The tiny experiments above show that these constraints are distinct, and that exchanges can improve balance without changing the first two.